# CPEC detection: evaluation and error analysis
Measures how the CPEC RetinaNet actually performs as a **detector**: precision (how many detections are real CPECs), recall (how many CPECs are found) and F1, at IoU ≥ 0.5, across score thresholds. The original notebook's median IoU only scores the boxes the model produced, so it can't see missed cells and mostly hides false positives.

Needs no retraining. Run it on your current model first to see which kind of error you have.

In [ ]:
import pickle, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tensorflow.keras import models
from jarvis.train.box import BoundingBox
import biondi
import cpec_tools as ct

## Paths
Use the **validation** set to choose thresholds and checkpoints; report final numbers on the **test** set once.

In [ ]:
model_path = ''          # trained model / checkpoint
eval_dataset_path = ''   # validation pickle (use test only for final numbers)

with open(eval_dataset_path, 'rb') as f:
    data = pickle.load(f)
model = models.load_model(model_path)

# must match the BoundingBox the dataset was encoded with
bb = BoundingBox(image_shape=(256, 256), classes=1, c=[3], anchor_shapes=[16],
                 anchor_scales=[1], anchor_ratios=[1], iou_upper=0.5, iou_lower=0.2)

## 1. Predict once, then sweep score thresholds

In [ ]:
preds = ct.predict_logits(model, data)
true_boxes = ct.ground_truth_boxes(bb, data)
cutoff = ct.find_decoder_cutoff(bb, preds)   # measured from jarvis, not assumed
print('decoder cutoff:', cutoff)

sweep = ct.threshold_sweep(bb, preds, true_boxes, cutoff=cutoff)
display(sweep.round(3))
ct.plot_sweep(sweep)
best = sweep.loc[sweep.f1.idxmax()]
print(f"Best F1 {best.f1:.3f} at score threshold {best.score_thr}  (P={best.precision:.3f}, R={best.recall:.3f})")

**Reading this:**
- **Low recall at every threshold** means the model misses CPECs. Check the anchor fit (section 2) and augmentation.
- **Low precision at every threshold** means the model fires on non-CPECs. Look at the false-positive tiles (section 3), then add hard negatives and augmentation.
- **Good F1 at some other threshold** means the default threshold was the problem. Use `best.score_thr` for whole-slide inference.

## 2. Do the anchors fit the cells?

In [ ]:
sizes = ct.box_size_report(true_boxes, anchor_size=16, anchor_scales=[1], anchor_ratios=[1], iou_upper=0.5)
# try a candidate configuration before retraining:
_ = ct.box_size_report(true_boxes, anchor_size=16, anchor_scales=[0.8, 1, 1.26, 1.6], anchor_ratios=[0.5, 1, 2])

## 3. Look at the errors
Green = correct, red = false positive, yellow dashed = missed CPEC.

In [ ]:
pred_boxes = ct.decode(bb, preds, best.score_thr, cutoff)
summary, per_tile = ct.detection_metrics(pred_boxes, true_boxes)
print(summary)
ct.show_tiles(data, pred_boxes, true_boxes, ct.worst_tiles(per_tile, 'fp', 12))   # false positives

In [ ]:
ct.show_tiles(data, pred_boxes, true_boxes, ct.worst_tiles(per_tile, 'fn', 12))   # missed CPECs

In [ ]:
per_tile.to_csv('cpec_per_tile_errors.csv', index=False)   # e.g. to pull hard-negative tiles

## 4. (Optional) choose the best checkpoint by F1

In [ ]:
chckpnt_dir = ''
results = ct.evaluate_checkpoints(sorted(glob.glob(chckpnt_dir + 'model_epoch*')), bb, data)
plt.plot(results.f1.values, marker='o'); plt.xlabel('checkpoint'); plt.ylabel('best F1'); plt.show()
print(results.loc[results.f1.idxmax(), 'checkpoint'])